In [ ]:
# title: MapBiomas Soil
# subtitle: 27. Filter SOC Design Matrix in GEE
# author: Alessandro Samuel-Rosa and Taciara Zborowski Horst
# data: 2025

In [21]:
# Load required libraries
import ee

# Initialize the Earth Engine API
# ee.Authenticate()
ee.Initialize(project='mapbiomas-solos-workspace')

# Define the path to the FeatureCollection directory
fc_dir = 'projects/mapbiomas-workspace/SOLOS/AMOSTRAS/MATRIZES/collection3/'
fc_name = 'c03_soc_v2025_11_23'
fc_path = f'{fc_dir}{fc_name}'
print(f'FeatureCollection path: {fc_path}')

# Load the FeatureCollection from the specified path
fc = ee.FeatureCollection(fc_path)

# Print the number of features in the FeatureCollection
num_features = fc.size().getInfo()
print(f'Number of features in the FeatureCollection: {num_features}')
# 27751 features out of 28065 in the original CSV. This means that 314 features had issues during
# generation of the design matrix in GEE.

# Print the columns of the FeatureCollection
print(fc.first().propertyNames().getInfo())

# Print the minimum and maximum values of the 'natural' property
print(fc.aggregate_min('natural').getInfo())
# Min: 0 (OK)
print(fc.aggregate_max('natural').getInfo())
# Max: 58 (OK)

FeatureCollection path: projects/mapbiomas-workspace/SOLOS/AMOSTRAS/MATRIZES/collection3/c03_soc_v2025_11_23
Number of features in the FeatureCollection: 27751
['Amazonas_Solimoes_Provincia', 'Vulcanicas_Subprovincia', 'year', 'HLZ_L2_Polar_Desert', 'Amazonia_Provincia', 'HLZ_L1_Subpolar_Zone', 'koppen_l2_Cf', 'koppen_l2_Cs', 'mb_ndvi_median_decay', 'HLZ_L2_Boreal_Moist_forest', 'Wetsols', 'cti', 'HLZ_L2_Subpolar_Dry_tundra', 'ARGISSOLO', 'ORGANOSSOLO', 'Savana_Estepica', 'id', 'koppen_l2_Cw', 'Plinthosols', 'Floresta_Ombrofila_Densa', 'Ferralsols', 'natural', 'Argisols', 'GLEISSOLO', 'HLZ_L2_Subtropical_Thorn_steppe', 'HLZ_L2_Tropical_Desert_bush', 'Borborema_Provincia', 'mb_savi_median_decay', 'HLZ_L2_Tropical_Wet_forest', 'Floresta_Estacional_Decidual', 'carbono_gm2', 'slope', 'Floresta_Ombrofila_Aberta', 'argila_000_030cm', 'Metamorficas_Subprovincia', 'Floresta_Estacional_Sempre_Verde', 'Formacao_Pioneira', 'PLANOSSOLO', 'eastness', 'formacaoSavanica', 'Parana_Provincia', 'HLZ_L2_

In [22]:
# Identify samples with land use/land cover classes
# We want to check if there are samples with no land use/land cover age assigned. This is
# necessary as the land use/land cover age is used as a time-wise predictor of SOC in the model.

# Consider the following properties of the FeatureCollection:
properties = ['campoAlagadoAreaPantanosa', 'formacaoCampestre', 'formacaoFlorestal',
              'formacaoSavanica', 'outrasFormacoesFlorestais', 'restingas', 'natural',
              'lavouras', 'mosaicoDeUsos', 'pastagem', 'silvicultura', 'antropico']

# Create a column that contains the sum of the values of the properties
fc = fc.map(lambda feature: feature.set('sum', ee.Number(0)))
for property in properties:
    fc = fc.map(lambda feature: feature.set('sum', ee.Number(feature.get('sum')).add(feature.get(property))))
                                  
# Print the minimum and maximum values of the 'sum' property
print(fc.aggregate_min('sum').getInfo())
# Min: 0 (OK)
print(fc.aggregate_max('sum').getInfo())
# 153

0
153


In [23]:
# Get the features where the sum of the values of the properties is equal to zero
# and print the result.
fc_filter = fc.filter(ee.Filter.eq('sum', 0))
print(fc_filter.size().getInfo())
# 503 features with no land use/land cover age.

# Get the 'id' column of the filtered FeatureCollection and print it
ids_no_lu_lc_age = fc_filter.aggregate_array('id').getInfo()
print(ids_no_lu_lc_age)

503
['ctb0787-46', 'ctb0787-46', 'ctb0787-57', 'ctb0787-57', 'ctb0661-P101', 'ctb0661-P101', 'ctb0661-P61', 'ctb0661-P61', 'ctb0661-P61', 'ctb0691-20', 'ctb0691-20', 'ctb0691-20', 'ctb0631-Perfil-45', 'ctb0631-Perfil-45', 'ctb0631-Perfil-45', 'ctb0631-Perfil-61', 'ctb0631-Perfil-61', 'ctb0631-Perfil-81', 'ctb0694-40', 'ctb0694-40', 'ctb0694-40', 'ctb0804-RS-16', 'ctb0804-RS-16', 'ctb0804-RS-47', 'ctb0804-RS-47', 'ctb0769-37', 'ctb0769-37', 'ctb0769-37', 'ctb0797-RS-166', 'ctb0797-RS-166', 'ctb0574-GB-42', 'ctb0574-GB-42', 'ctb0705-22', 'ctb0705-22', 'ctb0705-30', 'ctb0705-30', 'ctb0706-23', 'ctb0679-Perfil-327', 'ctb0679-Perfil-327', 'ctb0709-41', 'ctb0709-41', 'ctb0709-41', 'ctb0709-42', 'ctb0709-42', 'ctb0713-76', 'ctb0713-76', 'ctb0631-Perfil-25', 'ctb0631-Perfil-25', 'ctb0631-Perfil-25', 'ctb0642-Perfil-84', 'ctb0753-105', 'ctb0753-125', 'ctb0753-125', 'ctb0753-131', 'ctb0753-131', 'ctb0753-131', 'ctb0631-PC-122', 'ctb0631-PC-122', 'ctb0631-Perfil-17', 'ctb0631-Perfil-17', 'ctb0642

In [25]:
# There are the following cases:
# - Soil samples from before 1985 that fall on urban areas when mapped to 1985.
# - Recent soil samples collected close to urban areas (setttlements, roads, etc).
# - Soil samples collected close to water bodies or floodable areas.

# Show the spatial distribution of features with no land use/land cover age on the Brazilian map
import folium

m = folium.Map(location=[-14.2350, -51.9253], zoom_start=4)

# Add satellite imagery as base layer
folium.TileLayer(
  tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
  attr='Esri',
  name='Esri Satellite',
  overlay=False,
  control=True
).add_to(m)

# Add the filtered features layer
mapid = fc_filter.getMapId({'color': 'red'})
folium.TileLayer(
  tiles=mapid['tile_fetcher'].url_format,
  attr='Google Earth Engine',
  name='No Land Use/Land Cover Age',
  overlay=True,
  control=True
).add_to(m)

folium.LayerControl().add_to(m)
m


In [ ]:
# Check the following properties for low sample counts:
covariates = [
  # Pedologia IBGE: high sample counts.
  # 'PLANOSSOLO', 'LATOSSOLO', 'GLEISSOLO', 'NITOSSOLO', 'ARGISSOLO', 'CAMBISSOLO',
  # 'NEOSSOLO_LITOLICO', 'sibcs_rasos', 'sibcs_btextural', 'sibcs_esqueleto', 'sibcs_homogeneo',
  
  # Pedologia IBGE: low sample counts.
  'CHERNOSSOLO', 'PLINTOSSOLO', 'VERTISSOLO', 'LUVISSOLO', 'ESPODOSSOLO', 'ORGANOSSOLO',
  'NEOSSOLO_FLUVICO', 'NEOSSOLO_QUARTZARENICO', 'NEOSSOLO_REGOLITICO', 
  
  # clima Köppen: high sample counts.
  # 'koppen_l1_A', 'koppen_l2_Af', 'koppen_l2_Am',  'koppen_l2_As', 'koppen_l2_Aw', 'koppen_l1_C',
  # 'koppen_l2_Cf', 'koppen_l3_Cfa', 'koppen_l3_Cfb', 'koppen_l2_Cw', 'koppen_l3_Cwb', 
  
  # clima Köppen: low sample counts.
  'koppen_l1_B', 'koppen_l2_Bs', 'koppen_l3_Bsh', 'koppen_l3_Cwa', 'koppen_l3_Cwc', 'koppen_l2_Cs',
  'koppen_l3_Csa', 'koppen_l3_Csb',
  
  # biomas IBGE: high sample counts.
  # 'Amazonia', 'Caatinga', 'Cerrado', 'Mata_Atlantica', 'Pampa', 'Pantanal', 'Zona_Costeira',
  
  # fitofisionomias IBGE: high sample counts.
  # 'Estepe', 'Floresta_Estacional_Decidual', 'Floresta_Estacional_Semidecidual',
  # 'Floresta_Estacional_Sempre_Verde', 'Floresta_Ombrofila_Aberta', 'Floresta_Ombrofila_Densa',
  # 'Floresta_Ombrofila_Mista', 'Formacao_Pioneira', 'Savana', 'Savana_Estepica',
  
  # fitofisionomias IBGE: low sample counts.
  'Campinarana', 

  # províncias estruturais (IBGE): high sample counts.
  # 'Amazonia_Provincia', 'Cobertura_Cenozoica_Provincia', 'Costeira_Margem_Continental_Provincia',
  # 'Mantiqueira_Provincia', 'Parana_Provincia', 'Sao_Francisco_Provincia', 

  # províncias estruturais (IBGE): low sample counts.
  'Amazonas_Solimoes_Provincia', 'Borborema_Provincia', 'Gurupi_Provincia', 'Parecis_Provincia',
  'Parnaiba_Provincia', 'Reconcavo_Tucano_Jatoba_Provincia', 'Sao_Luis_Provincia',
  'Tocantis_Provincia',

  # subprovíncias: high sample counts.
  # 'Sedimentos_Subprovincia', 'Sedimentares_Subprovincia', 'Vulcanicas_Subprovincia',
  # 'Plutonicas_Subprovincia', 'Metamorficas_Subprovincia',
]
# As this are binary covariates, we will get the count of samples for each covariate by summing
# the values of the properties. Assuming that each soil profile has, on average, two samples, and
# targeting a minimum of 10 soil profiles per covariate, we will flag covariates with less than
# 20 samples.
for covariate in covariates:
    count = fc_filter.aggregate_sum(covariate).getInfo()
    if count < 20:
        print(f'Covariate: {covariate}, Count: {count}')


Covariate: CHERNOSSOLO, Count: 0
Covariate: PLINTOSSOLO, Count: 13
Covariate: VERTISSOLO, Count: 0
Covariate: LUVISSOLO, Count: 14
Covariate: ESPODOSSOLO, Count: 5
Covariate: ORGANOSSOLO, Count: 0
Covariate: NEOSSOLO_FLUVICO, Count: 9
Covariate: NEOSSOLO_QUARTZARENICO, Count: 14
Covariate: NEOSSOLO_REGOLITICO, Count: 0
Covariate: koppen_l1_B, Count: 19
Covariate: koppen_l2_Bs, Count: 19
Covariate: koppen_l3_Bsh, Count: 19
Covariate: koppen_l3_Cwa, Count: 15
Covariate: koppen_l3_Cwc, Count: 0
Covariate: koppen_l2_Cs, Count: 2
Covariate: koppen_l3_Csa, Count: 2
Covariate: koppen_l3_Csb, Count: 0
Covariate: Campinarana, Count: 0
Covariate: Amazonas_Solimoes_Provincia, Count: 18
Covariate: Borborema_Provincia, Count: 10
Covariate: Gurupi_Provincia, Count: 0
Covariate: Parecis_Provincia, Count: 10
Covariate: Parnaiba_Provincia, Count: 0
Covariate: Reconcavo_Tucano_Jatoba_Provincia, Count: 2
Covariate: Sao_Luis_Provincia, Count: 0
Covariate: Tocantis_Provincia, Count: 11
